In [2]:
import pandas as pd
import numpy as np
import glob
import polars as pl
from pathlib import Path
import dask.dataframe as dd
import altair as alt

DATA_PATH = Path("./data/UpdatedData/citibike_2023_combined.parquet/")

In [3]:
# 1) Read Parquet (all files or single file)
ddf = dd.read_parquet(f"{DATA_PATH}/*.parquet")

# 2) Parse datetimes
ddf["start_time"] = dd.to_datetime(ddf["started_at"])
ddf["end_time"]   = dd.to_datetime(ddf["ended_at"])

# 3) ride_time_seconds
ddf["ride_time_seconds"] = (ddf["end_time"] - ddf["start_time"]).dt.total_seconds()

# 4) Filter out null station names/ids
ddf = ddf[
    ddf["end_station_name"].notnull()
    & ddf["end_station_id"].notnull()
    & ddf["start_station_name"].notnull()
    & ddf["start_station_id"].notnull()
]

# 5) Filter ride_time between 1 minute and 2 hours
ddf = ddf[(ddf["ride_time_seconds"] < 2 * 3600) & (ddf["ride_time_seconds"] >= 60)]

# 6) Only year 2023 and add month
ddf = ddf[ddf["start_time"].dt.year == 2023]
ddf["month"] = ddf["start_time"].dt.month

# 7) Remove negative or zero durations
ddf = ddf[ddf["ride_time_seconds"] > 0]

# 8) Remove rides with identical start/end stations
ddf = ddf[ddf["start_station_id"] != ddf["end_station_id"]]
ddf = ddf[
    ~(
        (ddf["start_lat"] == ddf["end_lat"])
        & (ddf["start_lng"] == ddf["end_lng"])
    )
]

# 9) Select columns and sort only when materializing (optional)
cols = [
    "ride_id",
    "rideable_type",
    "start_time",
    "end_time",
    "ride_time_seconds",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual",
    "month",
]
ddf = ddf[cols]


In [4]:
# Add hour and month name (column-wise)
ddf["hour"] = ddf["start_time"].dt.hour

month_map = {
    1:"Jan",2:"Feb",3:"Mar",4:"Apr",
    5:"May",6:"Jun",7:"Jul",8:"Aug",
    9:"Sep",10:"Oct",11:"Nov",12:"Dec"
}
ddf["month_name"] = ddf["month"].map(month_map)

# Aggregate with Dask, then bring small result to pandas
agg = (
    ddf.groupby(["month_name", "hour", "member_casual"])
       .size()
       .compute()
       .reset_index(name="rides")
)

# Order months
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
agg["month_name"] = pd.Categorical(agg["month_name"], categories=month_order, ordered=True)
agg = agg.sort_values(["month_name", "hour"])

# Small multiples: facet by month, color by rider type
chart = (
    alt.Chart(agg)
    .mark_line()
    .encode(
        x=alt.X("hour:O", title="Hour of day"),
        y=alt.Y("rides:Q", title="Number of rides"),
        color=alt.Color("member_casual:N", title="Rider type"),
        facet=alt.Facet("month_name:N", columns=4, title="Month")
    )
    .properties(
        title="Hourly Citi Bike ridership by month and rider type (2023)"
    )
)

chart


c:\Users\Ankit P Bisleri\AppData\Local\Programs\Python\Python310\lib\site-packages\dask\dataframe\dask_expr\_collection.py:4215: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map function that you are using.
  Before: .map(func)
  After:  .map(func, meta=('month', 'object'))

  warnings.warn(meta_warning(meta, method="map"))


alt.Chart(...)

In [5]:
# Add hour and month name
ddf["hour"] = ddf["start_time"].dt.hour

month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}
ddf["month_name"] = ddf["month"].map(month_map)

# Aggregate
agg = (
    ddf.groupby(["month_name", "hour", "member_casual"])
       .size()
       .compute()
       .reset_index(name="rides")
)

# Order months
month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
agg["month_name"] = pd.Categorical(
    agg["month_name"], categories=month_order, ordered=True
)
agg = agg.sort_values(["month_name", "hour"])

# Small multiples: facet by month, sorted, smaller panels
chart = (
    alt.Chart(agg, width=120, height=90)  # shrink each facet
    .mark_line()
    .encode(
        x=alt.X("hour:O", title="Hour of day"),
        y=alt.Y("rides:Q", title="Number of rides"),
        color=alt.Color("member_casual:N", title="Rider type"),
        facet=alt.Facet("month_name:N",
                        columns=4,
                        title="Month",
                        sort=month_order)  # enforce order in facets
    )
    .properties(
        title="Hourly Citi Bike ridership by month and rider type (2023)"
    )
)

chart


c:\Users\Ankit P Bisleri\AppData\Local\Programs\Python\Python310\lib\site-packages\dask\dataframe\dask_expr\_collection.py:4215: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map function that you are using.
  Before: .map(func)
  After:  .map(func, meta=('month', 'object'))

  warnings.warn(meta_warning(meta, method="map"))


alt.Chart(...)

In [7]:
import altair as alt

base = (
    alt.Chart(agg)
    .mark_line()
    .encode(
        x=alt.X("hour:O", title="Hour of day", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("rides:Q", title="Number of rides"),
        color=alt.Color("member_casual:N", title="Rider type"),
    )
    .properties(
        width=130,   # a bit wider
        height=90    # adjust as needed
    )
)

chart = (
    base
    .facet(
        column=alt.Column("month_name:N", sort=month_order, title="Month")
    )
    .configure_axis(
        labelFontSize=8,
        titleFontSize=9,
        labelLimit=60  # allow more space for labels before clipping
    )
)

chart


alt.FacetChart(...)

In [19]:
import pandas as pd
import altair as alt

month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

# Ensure ordered categorical in pandas (good practice, but optional for Altair sorting)
agg["month_name"] = pd.Categorical(
    agg["month_name"],
    categories=month_order,
    ordered=True
)

base = (
    alt.Chart(agg)
    .mark_line()
    .encode(
        x=alt.X("hour:O", title="Hour of day", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("rides:Q", title="Number of rides"),
        color=alt.Color("member_casual:N", title="Rider type"),
    )
    .properties(width=130, height=90)
)

chart = (
    base
    .facet(
        facet=alt.Facet(
            "month_name:N",
            sort=month_order,   # ← explicit month order here
            title="Month"
        ),
        columns=4              # 4 per row → 3 rows total
    )
    .configure_axis(
        labelFontSize=8,
        titleFontSize=9,
        labelLimit=60
    )
    .properties(
        title="Hourly Citi Bike ridership by month and rider type (2023)"
    )
)

chart


alt.FacetChart(...)

In [29]:
import pandas as pd
import altair as alt

month_order = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

agg["month_name"] = pd.Categorical(
    agg["month_name"],
    categories=month_order,
    ordered=True
)

base = (
    alt.Chart(agg)
    .mark_line()
    .encode(
        x=alt.X("hour:O", title="Hour of day", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("rides:Q", title="Number of rides"),
        color=alt.Color("member_casual:N", title="Rider type"),
        tooltip=[
            alt.Tooltip("month_name:N", title="Month"),
            alt.Tooltip("hour:O", title="Hour"),
            alt.Tooltip("member_casual:N", title="Rider type"),
            alt.Tooltip("rides:Q", title="Rides", format=",")
        ],
    )
    .properties(width=210, height=90)
)

chart = (
    base
    .facet(
        facet=alt.Facet(
            "month_name:N",
            sort=month_order,
            title="Month"
        ),
        columns=4
    )
    .configure_axis(
        labelFontSize=8,
        titleFontSize=9,
        labelLimit=60
    )
    .properties(
        title="Hourly Citi Bike ridership by month and rider type (2023)"
    )
)

chart


alt.FacetChart(...)

The above has month by month comaprsion of number of rides in 2023

Seasonal and daily trends:
- Summer months (June–August) exhibit the highest overall ridership, peaking above 250,000 rides/hour during the evening rush.
- Winter months (December–February) see much lower volumes, typically under 150,000 rides/hour.
- Peak ridership consistently occurs at 8–9 AM and 5–7 PM, matching typical commute hours for members.

Member vs casual patterns:
- Members dominate total rides every month, with pronounced weekday commute peaks.
- Casual riders (blue line) have flatter patterns and contribute more on weekends and afternoons, especially in summer.

Insights:
- Citi Bike demand is strongly seasonal and closely follows NYC commuting patterns.
- Evening peaks are higher and wider than morning peaks, especially in warmer months.
- Leisure and tourist use (casual riders) rises sharply in summer, but never surpasses member commutes.

Ridership falls off sharply in winter, indicating weather’s strong impact on bike share usage.

Suggestions:
- Fleet planning and rebalancing should be adjusted for summer surges and winter declines.
- System is used overwhelmingly for commuting, but leisure traffic is significant in summer.

In [31]:
ddf["month"] = ddf["start_time"].dt.month

def month_to_season(m):
    if m in [12, 1, 2]:
        return "Winter"
    elif m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

# Map month → season
ddf["season"] = ddf["month"].map(month_to_season)

# 2) Aggregate by season, bike type, and rider type
season_bike = (
    ddf.groupby(["season", "rideable_type", "member_casual"])
       .size()
       .compute()
       .reset_index(name="rides")
)

# Order seasons
season_order = ["Winter", "Spring", "Summer", "Fall"]
season_bike["season"] = pd.Categorical(season_bike["season"], categories=season_order, ordered=True)
season_bike = season_bike.sort_values(["season", "rideable_type"])

c:\Users\Ankit P Bisleri\AppData\Local\Programs\Python\Python310\lib\site-packages\dask\dataframe\dask_expr\_collection.py:4215: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map function that you are using.
  Before: .map(func)
  After:  .map(func, meta=('month', 'object'))

  warnings.warn(meta_warning(meta, method="map"))


Bike Usage across seasons and labeled by Bike type

In [33]:
chart_season_bike = (
    alt.Chart(season_bike)
    .mark_bar()
    .encode(
        x=alt.X("season:N", sort=season_order, title="Season"),
        y=alt.Y("rides:Q", title="Number of rides"),
        color=alt.Color("rideable_type:N", title="Bike type"),
        column=alt.Column("member_casual:N", title="Rider type"),
        tooltip=["season", "member_casual", "rideable_type", "rides"]
    )
    .properties(
        title="Citi Bike usage by season, bike type, and rider group (2023)",
        width=210, height=100
    )
)

chart_season_bike

alt.Chart(...)

Bike Usage across seasons and labeled by Bike type in %

In [35]:
# Compute share within each season + rider type
season_bike_share = (
    season_bike
    .groupby(["season", "member_casual"])
    .apply(lambda g: g.assign(share=g["rides"] / g["rides"].sum()))
    .reset_index(drop=True)
)

chart_season_share = (
    alt.Chart(season_bike_share)
    .mark_bar()
    .encode(
        x=alt.X("season:N", sort=season_order, title="Season"),
        y=alt.Y("share:Q", axis=alt.Axis(format='%'), title="Share of rides"),
        color=alt.Color("rideable_type:N", title="Bike type"),
        column=alt.Column("member_casual:N", title="Rider type"),
        tooltip=["season", "member_casual", "rideable_type",
                 alt.Tooltip("share:Q", format=".1%")]
    )
    .properties(
        title="Share of classic vs e-bikes by season and rider group (2023)", width = 210, height = 190
    )
)

chart_season_share


alt.Chart(...)

Casual riders:
- In Winter, e‑bikes account for roughly 60–65% of all casual trips; classic bikes are ~35–40%.
- In Spring, the classic bike share increases, so e‑bikes drop to about 45%, classic bikes make up 55%.
- Summer sees the lowest e‑bike share for casuals, with classic bikes making up about 55–60% of trips.
- In Fall, the e‑bike share rises again, with e‑bikes representing almost 60% of casual trips.

Members:
- In Winter, e‑bikes are used for about 55% of member trips.
- Spring and Summer: classic bike share increases; each bike type is used for about half of rides (e‑bikes just under 50%).
- In Fall, the e‑bike share grows again to about 55% for members.

Insights:
- E‑bikes are consistently more popular among casual riders in Winter and Fall, making up the majority of trips in colder seasons.
- Classic bike share increases during warmer months (Spring/Summer) for both members and casuals, but the effect is more pronounced among casual riders.
- Among members, e‑bike use never drops much below 45%, indicating strong year-round demand for faster rides.

Casuals shift strongly toward classic bikes when weather is good, but prefer e‑bikes for convenience in less hospitable seasons. Members use both types more evenly, though e‑bikes still lead in Winter and Fall.

E‑bike and classic bike supplies may need to be rebalanced by season and by member/casual target segments.

## Compound Figures

In [21]:
# Haversine distance in km (vectorized for Dask via map_partitions)
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in km
    lat1 = np.radians(lat1.astype(float))
    lon1 = np.radians(lon1.astype(float))
    lat2 = np.radians(lat2.astype(float))
    lon2 = np.radians(lon2.astype(float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def add_distance(df_part):
    df_part = df_part.copy()
    df_part["distance_km"] = haversine_np(
        df_part["start_lat"],
        df_part["start_lng"],
        df_part["end_lat"],
        df_part["end_lng"],
    )
    return df_part

ddf = ddf.map_partitions(add_distance)

# Duration in minutes and speed (km per minute)
ddf["duration_min"] = ddf["ride_time_seconds"] / 60.0
ddf = ddf[ddf["duration_min"] > 0]

ddf["speed_km_per_min"] = ddf["distance_km"] / ddf["duration_min"]

# Clip extreme values to reduce outliers for visualization
ddf = ddf[(ddf["distance_km"] < 20) & (ddf["duration_min"] < 120)]

# Aggregate by bike type and rider type
agg = (
    ddf.groupby(["rideable_type", "member_casual"])
       .agg(
           avg_distance_km=("distance_km", "mean"),
           med_distance_km=("distance_km", "median"),
           avg_duration_min=("duration_min", "mean"),
           med_duration_min=("duration_min", "median"),
           avg_speed_km_min=("speed_km_per_min", "mean"),
           trip_count=("ride_id", "count"),
       )
       .compute()
       .reset_index()
)


In [ ]:
# Make each panel wider and taller
base_width = 200   # try 200–220 if you want even wider
base_height = 250

p1 = p1.properties(width=base_width, height=base_height)
p2 = p2.properties(width=base_width, height=base_height)
p3 = p3.properties(width=base_width, height=base_height)

compound_horizontal = alt.hconcat(
    p1,
    p2,
    p3,
    spacing=60   # increase horizontal space between panels
).resolve_scale(
    color="shared"
).configure_view(
    stroke=None
).properties(
    title="Ride characteristics by bike type and rider group (2023)"
)

compound_horizontal


alt.HConcatChart(...)

Avg duration (min):
- Classic bike trips last about 30–31 min for casual riders and ~12 min for members.
- Electric bike trips last about 29–30 min for casual riders and ~14 min for members.

Members’ trips are consistently shorter in duration than casual users, for both bike types.

Avg distance (km):
- Classic bike trips average ~3.5 km for casual riders, ~2 km for members.
- Electric bike trips average ~4.7 km for casual riders, ~2.3 km for members.

E-bikes support noticeably longer trips for both groups, with casuals going the furthest.

Avg speed (km/min):
- Classic bike: casual riders average about 0.13 km/min and members about 0.15 km/min.
- Electric bike: casuals average about 0.16 km/min and members about 0.21 km/min.

Electric bikes substantially increase average speed for both segments, but especially for members.

Key insights:
- Electric bikes encourage longer and faster trips compared to classic bikes for all user types.
- Casual riders take longer and go farther per trip than members, especially on e-bikes—suggesting leisure/tourist usage.
- Members’ average speeds and distances increase more with e-bikes than casuals, implying efficient commuting.

Implications:
- Casual users treat the system as a way to explore further, while members use e-bikes to save commute time.